## Uniform Distribution

### PDF: 

$$
p(x; a, b) = \frac{1}{b-a}
$$

The integrate of $p(x)$ shoulbe equals to 1

$$
\int_a^b p(x) dx= 1 
$$

### CDF:

$$
c(x; a, b) = \frac{1}{b - a}(x - a) 
$$

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import optimize
from Utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
%load_ext autoreload 
%autoreload 2

In [3]:
#utils conversions and constants

deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s

G_cu = G * (1/AU_m)**3 * (M_sun) * (year)**2  
mu = G_cu

print(f"G in Canonical units: {G_cu} AU^3/ (M_sun year^2)")
print(f"mu in Canonical units: {mu} AU^3/year^2")

G in Canonical units: 39.48894168123623 AU^3/ (M_sun year^2)
mu in Canonical units: 39.48894168123623 AU^3/year^2


ORBITAL ELEMENTS BOUNDS 

In [4]:
a_min = 0 #au
a_max = 2 #au
e_min = 0
e_max = 1
i_min = 0 #rad
i_max = np.pi #rad
Omega_min = 0 #rad
Omega_max = 2*np.pi #rad
w_min = 0 #rad
w_max = 2*np.pi #rad
E_min = 0 #rad
E_max = 2*np.pi #rad

p_E = p_E_uniform(a_min, a_max, e_min, e_max, i_min, i_max, Omega_min, Omega_max, w_min, w_max, E_min, E_max)

print(f'Probability uniform distribution for orbital elements: ', p_E)

Probability uniform distribution for orbital elements:  0.000641623890917771


### Possition space volume

In [5]:
x = 1  #AU
y = 0
z = 0
center = (x,y,z)

delta_r = 0.1  #AU
delta_x = delta_r
delta_y = delta_r
delta_z = delta_r
dimensions = (delta_x, delta_y, delta_z)

corners = volume_3d(center, dimensions)

In [6]:
x, y, z = corners[:,0], corners[:,1], corners[:,2]
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z,
                                   mode='markers')])
fig.add_trace(go.Scatter3d(x=[center[0]], y=[center[1]], z=[center[2]],
            mode='markers',
            name='center'))
fig.show()

Velocity Volume


In [7]:
x = 1  #AU 
vx = 0
vy = (mu/x)**0.5
vz = 0
center_v = (vx,vy,vz)

delta_v = 5000 * (1/AU_m) * year #AU/year
delta_vx = delta_v
delta_vy = delta_v
delta_vz = delta_v

dimensions_v = (delta_vx, delta_vy, delta_vz)

corners_v = volume_3d(center_v, dimensions_v)

In [9]:
delta_v

1.0547326203208556

In [7]:
x_v, y_v, z_v = corners_v[:,0], corners_v[:,1], corners_v[:,2]
fig = go.Figure(data=[go.Scatter3d(x=x_v, y=y_v, z=z_v,
                                   mode='markers')])
fig.add_trace(go.Scatter3d(x=[center_v[0]], y=[center_v[1]], z=[center_v[2]],
            mode='markers',
            name='center'))
fig.show()

In [8]:
a = a_max 
e = e_max - 0.1
i = i_max
Omega = Omega_max
w = w_max
M = 2*np.pi
q = a*(1-e)

elements_spice = np.array([q, e, i, Omega, w, M, 0.0, mu])
print('elements spice: ', elements_spice)

et = 0 
state_vector_spice = spy.conics(elements_spice, et)
print('state vector using spice: ', state_vector_spice)

elements spice:  [ 0.2         0.9         3.14159265  6.28318531  6.28318531  6.28318531
  0.         39.48894168]
state vector using spice:  [ 2.00000000e-01  0.00000000e+00 -5.99903913e-33  0.00000000e+00
 -1.93686589e+01  2.37197661e-15]


In [9]:
#check if state vector is inside volume
volume_object = is_point_in_volume(state_vector_spice[:3], state_vector_spice[3:], center, center_v, dimensions, dimensions_v)
print('check if state position vector is inside volume: ', volume_object)

check if state position vector is inside volume:  False


generar elementos orbitales 

In [10]:
N = int(1e8)

file_state = f"state_vector_N{N}.dat"

if not os.path.isfile(file_state) or 0:
    print('Generating state vector')
    a_uniform = np.random.uniform(0, 2, N)
    e_uniform = np.random.uniform(0, 1, N)
    i_uniform = np.random.uniform(0, 2*np.pi, N)
    w_uniform = np.random.uniform(0, 2*np.pi, N)
    Omega_uniform = np.random.uniform(0, 2*np.pi, N)
    M_uniform = np.random.uniform(0, 2*np.pi, N)

    state_vector = np.zeros((N, 6))
    for element in tqdm(range(N)):
        a = a_uniform[element]
        e = e_uniform[element]
        i = i_uniform[element]
        Omega = Omega_uniform[element]
        w = w_uniform[element]
        M = M_uniform[element]
        q=a*(1-e)

        elements_spice = np.array([q, e, i, Omega, w, M, 0.0, mu])
        et = 0 
        state_vector_spice = spy.conics(elements_spice, et)
        state_vector[element] = state_vector_spice

    np.savetxt(file_state, state_vector)
else:
    print('state vector already generated')
    state_vector = np.loadtxt(file_state)

state vector already generated


In [11]:
state_vector

array([[-0.40595751, -0.54605137, -0.40531802,  3.19632734, -1.1950685 ,
        -6.16572945],
       [ 0.03181708,  1.10535452,  0.22687246, -3.09328008, -2.40188736,
         5.50952744],
       [-0.01942426, -0.14019779, -0.28246791,  0.60527281,  7.6759661 ,
        12.25627549],
       ...,
       [ 1.85632302, -0.73460863, -0.46220131,  1.86307423,  3.31543758,
        -1.34266731],
       [ 0.70876312,  0.36811929,  0.29243071, -2.57651748,  6.29320845,
        -0.01697366],
       [-1.20240298,  0.72364986, -0.55580532, -4.64275263, -1.32160345,
        -0.2755162 ]], shape=(100000000, 6))

In [12]:
print(f'Volume center: {center, center_v}')
print(f'Dimensions: {delta_x, delta_y, delta_z, delta_vx, delta_vy, delta_vz}')

objsx = (abs(state_vector[:,0] - center[0]) <= delta_x/2) 
objsy = (abs(state_vector[:,1] - center[1]) <= delta_y/2) 
objsz = (abs(state_vector[:,2] - center[2]) <= delta_y/2) 
objsvx = (abs(state_vector[:,3] - center_v[0]) <= delta_vx/2) 
objsvy = (abs(state_vector[:,4] - center_v[1]) <= delta_vy/2) 
objsvz = (abs(state_vector[:,5] - center_v[2]) <= delta_vz/2) 

objects = objsx * objsy * objsz * objsvx * objsvy * objsvz
print(f'Number of objects inside volume: {objects.sum()}')

Volume center: ((1, 0, 0), (0, 6.2840227308020005, 0))
Dimensions: (0.1, 0.1, 0.1, 1.0547326203208556, 1.0547326203208556, 1.0547326203208556)
Number of objects inside volume: 247


In [13]:
select_state_vector = state_vector[objects]
select_state_vector

array([[ 1.02111039e+00,  1.65876416e-02, -1.81726913e-03,
        -3.12519234e-01,  5.99426389e+00,  7.86392948e-03],
       [ 9.96986705e-01,  1.51681106e-02,  7.61873442e-03,
        -2.17876848e-01,  5.96734721e+00, -4.49082084e-01],
       [ 1.01713106e+00, -9.48300557e-04, -1.33637772e-02,
         6.91126292e-02,  5.89884081e+00,  7.43742203e-02],
       ...,
       [ 1.02954870e+00, -1.16781572e-02, -2.80510917e-02,
        -1.23972933e-01,  6.32522981e+00,  4.83117287e-01],
       [ 1.04459686e+00, -1.42758370e-02, -1.54036026e-02,
        -5.14183279e-02,  6.11157934e+00, -1.82344794e-01],
       [ 9.55043589e-01, -1.54232815e-02, -1.83645028e-02,
         2.76547245e-01,  6.55582953e+00, -2.75366377e-01]],
      shape=(247, 6))

In [14]:
fig = make_subplots(rows=1, cols=2,  specs=[[{'type': 'scene'}, {'type': 'scene'}]])

x, y, z = corners[:,0], corners[:,1], corners[:,2]
vx, vy, vz = corners_v[:,0], corners_v[:,1], corners_v[:,2]

fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='markers'), row=1, col=1)

fig.add_trace(go.Scatter3d(x=[center[0]], y=[center[1]], z=[center[2]],
            mode='markers',
            name='center'), row=1, col=1)

fig.add_trace(go.Scatter3d(x=list(select_state_vector[:,0]), 
                           y=list(select_state_vector[:,1]), 
                           z=list(select_state_vector[:,2]),
            mode='markers',
            name='center', 
            marker=dict(
                color='LightSkyBlue',
                size=4
            )
        ), row=1, col=1)

fig.add_trace(go.Scatter3d(x=vx, y=vy, z=vz,
                            mode='markers'), row=1, col=2)

fig.add_trace(go.Scatter3d(x=[center_v[0]], y=[center_v[1]], z=[center_v[2]],
            mode='markers',
            name='center', 
            marker=dict(
                color='black',
                size=4
            )
        ), row=1, col=2)

fig.add_trace(go.Scatter3d(x=list(select_state_vector[:,3]), 
                            y=list(select_state_vector[:,4]), 
                            z=list(select_state_vector[:,5]),
            mode='markers',
            name='center', 
            marker=dict(
                color='LightSkyBlue',
                size=4
            )
        ), row=1, col=2)
        
fig.show()    

## Probability

\begin{equation}
    p(\widetilde{X})|d\widetilde{X}| = \eta(\widetilde{\varepsilon})|d\widetilde{\varepsilon}|
\end{equation}

\begin{equation}
    p(\widetilde{X}) = \eta(\widetilde{\varepsilon})|\frac{d\widetilde{\varepsilon}}{d\widetilde{X}}|
\end{equation}

\begin{equation}
    p(\widetilde{X}) = \eta(\widetilde{\varepsilon}(\widetilde{X}))\text{ det }\mathbb{J}_{\varepsilon \widetilde{X}}
\end{equation}

\begin{equation}
    \mathbb{J}_{\varepsilon \widetilde{X}} \equiv \frac{\partial \widetilde{\varepsilon}}{\partial \widetilde{X}} \equiv 
    \begin{pmatrix} 
    \partial_xa & \partial_ya & \partial_za & \partial_{v_x}a  & \partial_{v_y}a & \partial_{v_z}a\\
    \partial_xe & \partial_ye & \partial_ze & \partial_{v_x} e & \partial_{v_y}e & \partial_{v_z}e \\
    \partial_xi & \partial_yi & \partial_zi & \partial_{v_x} i & \partial_{v_y}i & \partial_{v_z}i \\
    \partial_x \Omega & \partial_y \Omega & \partial_z \Omega & \partial_{v_x} \Omega & \partial_{v_y} \Omega & \partial_{v_z } \Omega \\
    \partial_xw & \partial_yw & \partial_zw & \partial_{v_x} w & \partial_{v_y}w & \partial_{v_z}w  \\
    \partial_xM & \partial_yM & \partial_zM & \partial_{v_x} M & \partial_{v_y}M & \partial_{v_z}M
    \end{pmatrix}
\end{equation}

In this case we have

\begin{equation}
    \mathbb{J}_{X \varepsilon } \equiv \frac{\partial \widetilde{X}}{\partial \widetilde{\varepsilon}} \equiv 
    \begin{pmatrix} 
    \partial_ax & \partial_ex & \partial_ix & \partial_\Omega x & \partial_wx & \partial_Mx\\
    \partial_ay & \partial_ey & \partial_iy & \partial_\Omega y & \partial_wy & \partial_My \\
    \partial_az & \partial_ez & \partial_iz & \partial_\Omega z & \partial_wz & \partial_Mz \\
    \partial_av_x & \partial_ev_x & \partial_iv_x & \partial_\Omega v_x & \partial_wv_x & \partial_Mv_x \\
    \partial_av_y & \partial_ev_y & \partial_iv_y & \partial_\Omega v_y & \partial_wv_y & \partial_Mv_y \\
    \partial_av_z & \partial_ev_z & \partial_iv_z & \partial_\Omega v_z & \partial_wv_z & \partial_Mv_z
    \end{pmatrix}
\end{equation}

So we can use this property: 

\begin{equation}
    \mathbb{J}_{\varepsilon \widetilde{X}} = \mathbb{J}_{X \varepsilon }^{-1}
\end{equation}

\begin{equation}
    \det \mathbb{J}_{\varepsilon \widetilde{X}} = \frac{1}{\det \mathbb{J}_{X \varepsilon}} 
\end{equation}

Point $X_0 -> \varepsilon_0$

In [15]:
select_orbital_elements = spy.oscelt(select_state_vector[2], et=0, mu=mu)
print(f"State vector selected: {select_state_vector[2]}")
print(f"Orbital elements calculated: {select_orbital_elements}")

State vector selected: [ 1.01713106e+00 -9.48300557e-04 -1.33637772e-02  6.91126292e-02
  5.89884081e+00  7.43742203e-02]
Orbital elements calculated: [8.26083683e-01 1.03932906e-01 1.83059697e-02 7.99485295e-01
 2.43279487e+00 3.02926441e+00 0.00000000e+00 3.94889417e+01]


### Jaccobian Matrix

In [16]:
q = select_orbital_elements[0]
e = select_orbital_elements[1]
i = select_orbital_elements[2]
Omega = select_orbital_elements[3]
w = select_orbital_elements[4]
M = select_orbital_elements[5]
E = optimize.newton(Kepler, 1, args=(M, e))
a = q/(1-e)

elements = OrbitalElements(a, e, i, Omega, w, E)
grav_params = GravitationalParameters(mu=mu)

J_XE = JaccobianComponents.Jacobian(elements, grav_params)
J_XE


array([[ 1.10329945e+00,  9.35110240e-01, -9.58179347e-03,
         9.48300557e-04,  1.11866262e-03,  9.73520137e-03],
       [-1.02863783e-03, -9.03791711e-03,  9.31556626e-03,
        -1.01713106e+00,  1.01713603e+00,  8.30910412e-01],
       [-1.44959176e-02, -1.23903467e-02, -7.29941360e-01,
         0.00000000e+00,  1.29660817e-02,  1.04763488e-02],
       [-3.74838252e-02, -1.07663426e+00,  5.33261224e-02,
        -5.89884081e+00, -5.89880148e+00, -5.37521914e+00],
       [-3.19928674e+00, -1.64190046e-01, -5.18444724e-02,
         6.91126292e-02,  6.81249175e-02,  5.01147149e-03],
       [-4.03374942e-02,  1.20373558e-02,  4.06238586e+00,
         0.00000000e+00,  7.83017962e-02,  7.06233777e-02]])

Inverse Jacobian Matrix

In [17]:
J_EX = JaccobianComponents.Jacobian_inv(elements, grav_params)
J_EX

array([[-4.68862330e+00,  4.72627854e+00, -3.55561334e+02,
        -8.09900150e-01,  4.95117314e-01, -6.38932460e+01],
       [ 4.27047586e+00, -3.19469127e+00,  2.40324467e+02,
         5.49597007e-01, -1.66165755e-01,  4.31902656e+01],
       [-2.34517349e-01,  2.28001357e-01, -1.78655398e+01,
        -3.89085424e-02,  3.78274804e-02, -2.96405411e+00],
       [-1.21690646e+01,  1.18309509e+01, -9.27039765e+02,
        -2.18657308e+00,  2.12581981e+00, -1.66573213e+02],
       [-2.16094701e+02,  2.21194159e+02, -1.66036512e+04,
        -3.77216523e+01,  3.86955391e+01, -2.98342048e+03],
       [ 2.49673284e+02, -2.55114216e+02,  1.91924707e+04,
         4.35047105e+01, -4.47674432e+01,  3.44859071e+03]])

Determinant Inverse Jacobian Matrix

In [18]:
det = np.abs(np.linalg.det(J_EX))
det

np.float64(140.75528088312413)

$$ n = Np(\widetilde{X})|\Delta \widetilde{X}| $$
$$ |\Delta \widetilde{X}| = \Delta x \Delta y \Delta z \Delta v_x \Delta v_y \Delta v_z $$

In [19]:
Delta = delta_x/2 * delta_y/2 * delta_z/2 * delta_vx/2 * delta_vy/2 * delta_vz/2
Delta

1.833357500704041e-05

In [20]:
n = N * p_E * det * Delta
n

np.float64(165.57409274606425)

### Analytical formula from professor

In [22]:
q = select_orbital_elements[0]
e = select_orbital_elements[1]
i = select_orbital_elements[2]
Omega = select_orbital_elements[3]
w = select_orbital_elements[4]
M = select_orbital_elements[5]
a = q/(1-e)

E = [q, e, i, Omega, w, M]
X = spy.conics(list(E)+[0, mu], 0)
JXoE = calcKeplerianJacobians(mu,E,X)
JXoE

array([[ 1.10329945e+00,  9.18142477e-01, -9.58179347e-03,
         9.48300557e-04,  1.11866262e-03,  9.73520137e-03],
       [-1.02863783e-03,  1.77721772e-01,  9.31556626e-03,
         1.01713106e+00,  1.01713603e+00,  8.30910412e-01],
       [-1.44959176e-02, -9.78417313e-03, -7.29941360e-01,
         0.00000000e+00,  1.29660817e-02,  1.04763488e-02],
       [-3.74838252e-02, -5.52613171e-01,  5.33261224e-02,
        -5.89884081e+00, -5.89880148e+00, -5.37521914e+00],
       [-3.19928674e+00, -5.93195173e+00, -5.18444724e-02,
         6.91126292e-02,  6.81249175e-02,  5.01147149e-03],
       [-4.03374942e-02, -6.84499699e-02,  4.06238586e+00,
         0.00000000e+00,  7.83017962e-02,  7.06233777e-02]])

In [23]:
JEoX = np.linalg.inv(JXoE)
JEoX

array([[ 1.64259380e+00, -1.53143747e-03, -2.15815429e-02,
         2.97494503e-03,  2.53914912e-01,  3.20142960e-03],
       [-8.76582659e-01,  1.00142318e-01,  1.27847481e-02,
         1.35823949e-02, -3.02494833e-01, -4.03875507e-03],
       [-9.35552568e-03,  9.09558526e-03, -7.12704272e-01,
         1.54919193e-03, -1.50614810e-03,  1.18017495e-01],
       [-4.85456605e-01,  4.71968342e-01, -3.69821011e+01,
        -8.72282611e-02,  8.48046505e-02, -6.64505197e+00],
       [ 1.36183155e+00,  8.93631283e+00,  3.70844679e+01,
         1.54340722e+00,  1.97080084e-01,  6.62842068e+00],
       [-8.83164219e-01, -1.03349087e+01, -1.20301759e-01,
        -1.78546057e+00, -2.80029664e-01,  1.98636224e-02]])

In [24]:
det = np.abs(np.linalg.det(JEoX)) 
det

np.float64(4.412182264340108)

In [25]:
n = N * p_E * det * Delta
n

np.float64(5.190164595351799)

### Numerical Jacobian Calculation 

In [26]:
def computeNumericalJacobian(jfun,x,dx,**args):
    """
    Computes numerically the Jacobian matrix of a multivariate function.

    Parameters:
        jfun: multivariate function with the prototype "def jfun(x,**args)", function
        x: indepedent variables, numpy array (N).
        dx: step size of independent variables, numpy array (N).
        **args: argument of the function

    Return:
        y: dependent variables, y=jfun(x,**args)
        Jyx: Jacobian matrix:

            Jif= [dy_1/dx_1,dy_1/dx_2,...,dy_1/dx_N,
                dy_2/dx_1,dy_2/dx_2,...,dy_2/dx_N,
                                . . .
                dy_N/dx_1,dy_N/dx_2,...,dy_N/dx_N,]
    """
    N=len(x)
    J=np.zeros((N,N))
    y=jfun(x,**args)
    for i in range(N):
        for j in range(N):
            pre=[x[k] for k in range(j)]
            pos=[x[k] for k in range(j+1,N)]
            yi=lambda t:jfun(pre+[t]+pos,**args)[i]
            dyidxj=(yi(x[j]+dx[j])-yi(x[j]-dx[j]))/(2*dx[j])
            J[i,j]=dyidxj
    return y,J

In [27]:
def X2E(X,mu):
    elts=spy.oscelt(X,0,mu)
    E=elts[:6]
    return E

def E2X(E,mu):
    E = E + [0, mu]
    elts=spy.conics(E,0)
    X=elts[:6]
    return X 

In [28]:
q = select_orbital_elements[0]
e = select_orbital_elements[1]
i = select_orbital_elements[2]
Omega = select_orbital_elements[3]
w = select_orbital_elements[4]
M = select_orbital_elements[5]
a = q/(1-e)

E = [q, e, i, Omega, w, M] 
X = spy.conics(E + [0, mu],0)
dX=np.array([1e-3]*6)
args=dict(mu=mu)
E_num, JEoX_num=computeNumericalJacobian(X2E,X,dX,**args)

E_num, JEoX_num

(array([0.82608368, 0.10393291, 0.01830597, 0.79948529, 2.43279487,
        3.02926441]),
 array([[ 2.27999697e+00, -9.36897502e-02, -3.11247852e-02,
         -9.85583812e-03,  5.06394641e-01,  6.59202178e-03],
        [-8.76582323e-01,  1.00138267e-01,  1.27847461e-02,
          1.35823805e-02, -3.02494821e-01, -4.03875511e-03],
        [-9.35553786e-03,  9.09558430e-03, -7.12216213e-01,
          1.54919192e-03, -1.50614816e-03,  1.18014887e-01],
        [-4.85456675e-01,  4.71968418e-01, -3.70212680e+01,
         -8.72282615e-02,  8.48046510e-02, -6.64523046e+00],
        [ 1.36189368e+00,  8.93608249e+00,  3.71236351e+01,
          1.54340621e+00,  1.97082598e-01,  6.62859918e+00],
        [-8.83226273e-01, -1.03346782e+01, -1.20302108e-01,
         -1.78545957e+00, -2.80032179e-01,  1.98636256e-02]]))

In [29]:
det = np.abs(np.linalg.det(JEoX_num))
det

np.float64(3.9540627137716653)

In [30]:
Delta = 2*delta_x * 2*delta_y * 2*delta_z * 2*delta_vx * 2*delta_vy * 2*delta_vz
Delta

0.07509432322883752

In [31]:
n = N * p_E * det * Delta
n

np.float64(19051.58827728255)